# Busca de Correspondências em nomes de obras em duas bases

In [12]:
import pandas as pd
import pdfplumber
import re

### Subsecção: Pegando os nomes das obras que vão para a exposição na Cinha, a partir do pdf: 

In [3]:
import pdfplumber
import re

# Caminhos dos arquivos (ajuste conforme necessário)
caminho_pdf = "Textos_Curatoriais_China.pdf"
caminho_saida_txt = "nomes_obras_China.txt"

nomes_obras = []

with pdfplumber.open(caminho_pdf) as pdf:
    for pagina in pdf.pages:
        texto = pagina.extract_text()
        
        if texto:
            # O Regex r'\[(.*?)\]' busca tudo o que está dentro de []
            encontrados = re.findall(r'\[(.*?)\]', texto)
            
            for item in encontrados:
                # Limpa espaços em branco nas pontas
                item_limpo = item.strip()
                
                # Ignora as marcações automáticas como "Image 1", "Image 2", etc.
                if not item_limpo.lower().startswith('image'):
                    nomes_obras.append(item_limpo)

# Salvar a lista em um arquivo de texto, um nome por linha
with open(caminho_saida_txt, 'w', encoding='utf-8') as f:
    for nome in nomes_obras:
        f.write(f"{nome}\n")

print(f"{len(nomes_obras)} nomes de obras foram extraídos e salvos em: {caminho_saida_txt}")

67 nomes de obras foram extraídos e salvos em: nomes_obras_China.txt


In [25]:
# 1. Carregar o Dataframe
df_poemas = pd.read_csv('poemas_com_obras.csv')

# 2. Carregar a lista e já normalizar (minúsculas e sem espaços nas pontas)
nomes_obras_china_normalizados = []
with open('nomes_obras_China.txt', 'r', encoding='utf-8') as f:
    for linha in f:
        # Pula linhas vazias, se houver
        if linha.strip():
            nome_normalizado = linha.strip().lower()
            nomes_obras_china_normalizados.append(nome_normalizado)

print(f"Total de nomes únicos no PDF: {len(set(nomes_obras_china_normalizados))}")

# nomes repetidos:
from collections import Counter
contador = Counter(nomes_obras_china_normalizados)
repetidos = {nome: contagem for nome, contagem in contador.items() if contagem > 1}
print(f"Nomes repetidos: {repetidos}")

Total de nomes únicos no PDF: 67
Nomes repetidos: {'crianças brincando': 2, 'retirantes': 2}


## Comparando

In [26]:
# 3. Anotar os poemas que têm correspondência
obras_correspondentes_china = []

for index, row in df_poemas.iterrows():
    # Pega o título no dataframe, transforma em string para evitar erros (caso seja NaN)
    # e aplica a mesma normalização: minúsculas e sem espaços
    nome_obra = str(row['titulo']).strip().lower()
    
    # Compara a versão normalizada do CSV com a lista normalizada do TXT
    if nome_obra in nomes_obras_china_normalizados:
        # Se encontrou, salva o nome original (como está no CSV) para manter a formatação
        obras_correspondentes_china.append(row['titulo'])

print("\nObras encontradas (formato original do CSV):")
for obra in obras_correspondentes_china:
    print(f"- {obra}")

print(f"\nTotal de correspondências encontradas: {len(obras_correspondentes_china)}")


Obras encontradas (formato original do CSV):
- Circo
- Café
- Moça
- Flores
- Circo
- Menino de Brodowski
- Crianças brincando
- Enterro
- Retirantes
- Criança morta
- Menino com carneiro
- Denise com carneiro branco

Total de correspondências encontradas: 12


## Conferência manual
#### Obras correspondentes:
- Circo, 1933
- Moça, 1940
- Criança Morta, 1944
- Menino com Carneiro, 1954
- Denise com Carneiro, 1961